# Clustering de clientes con K-Means

In [ ]:
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
df = pd.read_csv('./data/retailmax.csv')
df.head()

## Selección de features
Usamos edad, ingreso anual y spending score. CustomerID no aporta información y Gender es categórica (la agregamos al final).

In [ ]:
features = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]
features.sample(5, random_state=42)

## Estandarización
Dejamos todas las variables con media 0 y desviación estándar 1 para que ninguna domine las distancias.

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

print(type(scaled_features))
print(scaled_features.shape)
scaled_features[0:3]

## Primeras pruebas con 3, 5 y 10 clusters

In [ ]:
for k in [3, 5, 10]:
    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
    df['Cluster'] = kmeans.fit_predict(scaled_features)

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', palette='Set1')
    plt.title(f'Clusters de clientes (k={k})')
    plt.show()

## Método del codo
Graficamos la WCSS para k de 1 a 10 y buscamos el punto donde la curva se "dobla".

In [ ]:
def calcular_wcss(datos):
    wcss = []
    for n in range(1, 11):
        kmeans = KMeans(n_clusters=n, init='k-means++', max_iter=300, n_init=10, random_state=42)
        kmeans.fit(datos)
        wcss.append(kmeans.inertia_)
    return wcss

wcss = calcular_wcss(scaled_features)

plt.plot(range(1, 11), wcss, marker='o')
plt.title('Método del codo')
plt.xlabel('Número de clusters')
plt.ylabel('WCSS')
plt.show()

El codo aparece alrededor de **5 clusters**, así que usamos k=5.

## Modelo final con 5 clusters

In [ ]:
kmeans = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(scaled_features)

df['Cluster'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', palette='Set1')
plt.title('Clusters de clientes')
plt.show()

## Visualización 3D

In [ ]:
sns.set(style='whitegrid')

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], df['Age'],
                     c=df['Cluster'], cmap='Set1', s=50)
ax.set_xlabel('Annual Income (k$)')
ax.set_ylabel('Spending Score (1-100)')
ax.set_zlabel('Age')
ax.set_title('Clusters de clientes en 3D')
legend1 = ax.legend(*scatter.legend_elements(), title='Clusters')
ax.add_artist(legend1)
plt.show()

## Análisis de clusters

In [ ]:
cluster_analysis = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()
cluster_analysis

In [ ]:
for c in cluster_analysis.index:
    print(f"Cluster {c}:")
    print(f" - Edad promedio: {cluster_analysis.loc[c, 'Age']:.2f}")
    print(f" - Ingreso anual promedio: {cluster_analysis.loc[c, 'Annual Income (k$)']:.2f}k$")
    print(f" - Spending Score promedio: {cluster_analysis.loc[c, 'Spending Score (1-100)']:.2f}")
    print("================================")

## Incorporando la variable sexo

In [ ]:
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

extended_features = df[['Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']]
scaled_extended = scaler.fit_transform(extended_features)

kmeans_ext = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=42)
df['Cluster'] = kmeans_ext.fit_predict(scaled_extended)

ext = df.groupby('Cluster')[['Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()
ext

In [ ]:
for c in ext.index:
    print(f"Cluster {c}:")
    print(f" - Edad promedio: {ext.loc[c, 'Age']:.2f}")
    print(f" - Ingreso anual promedio: {ext.loc[c, 'Annual Income (k$)']:.2f}k$")
    print(f" - Spending Score promedio: {ext.loc[c, 'Spending Score (1-100)']:.2f}")
    print(f" - % mujeres: {ext.loc[c, 'Gender']*100:.2f}%")
    print("================================")

## Conclusiones
- Se identificaron 5 segmentos: premium (ingreso y gasto altos), ahorradores con dinero (ingreso alto, gasto bajo), jóvenes entusiastas (ingreso medio-bajo, gasto alto), estables mayores (ingreso y gasto medios) y bajo ingreso/bajo gasto.
- Al agregar el sexo, los grupos de mayores de 50 se separan por hombres y mujeres, lo que sugiere campañas diferenciadas para ese rango de edad.